# DINOv3埋め込みによる机上異常検出

このNotebookは、論文 `Real-World On-Vehicle Evaluation of Embedding-Based Anomaly Detection` のコア手法を、机上画像で試すための静止画像版です。

本線ではDINOv3を固定特徴抽出器として使い、1枚の正常基準画像 `reference_normal` と検査画像 `observed` をpatch embeddingで比較します。検査画像の各patchについて、正常画像patchの最近傍コサイン類似度を取り、似ていない場所を異常スコアにします。

DINOv2は最後の付録に分離しています。DINOv3本線の結果とは混ぜません。

In [ ]:
%pip install -U "transformers>=4.56.0" huggingface_hub torch torchvision pillow matplotlib pandas tqdm numpy scipy

## 1. ライブラリ読み込み

GPUが使える場合は自動でCUDAを使います。

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

import torch
import torch.nn.functional as F
from torchvision import transforms

from scipy import ndimage

import matplotlib.pyplot as plt
from matplotlib import colormaps
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    display = print

torch.set_grad_enabled(False)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

## 2. ディレクトリ設定

Notebookを `notebooks/` の中で開いている前提です。

In [ ]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
REFERENCE_DIR = DATA_DIR / "reference_normal"
OBSERVED_DIR = DATA_DIR / "observed"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "desk_anomaly_dino"
HEATMAP_DIR = OUTPUT_DIR / "heatmap"
OVERLAY_DIR = OUTPUT_DIR / "overlay"
MASK_DIR = OUTPUT_DIR / "mask"
ANNOTATED_DIR = OUTPUT_DIR / "annotated"
COMPARISON_DIR = OUTPUT_DIR / "comparison"
COMPONENTS_DIR = OUTPUT_DIR / "components"
CACHE_DIR = PROJECT_ROOT / "cache" / "reference_features"

for d in [
    REFERENCE_DIR,
    OBSERVED_DIR,
    HEATMAP_DIR,
    OVERLAY_DIR,
    MASK_DIR,
    ANNOTATED_DIR,
    COMPARISON_DIR,
    COMPONENTS_DIR,
    CACHE_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REFERENCE_DIR:", REFERENCE_DIR)
print("OBSERVED_DIR:", OBSERVED_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 3. 画像確認

`reference_normal` には正常な机、`observed` には検査したい現在画像を入れます。最初に入力画像を横並びで確認します。

In [ ]:
IMAGE_EXTENSIONS = ["*.jpg", "*.jpeg", "*.png", "*.webp", "*.JPG", "*.JPEG", "*.PNG", "*.WEBP"]

def list_images(folder: Path):
    files = []
    for pattern in IMAGE_EXTENSIONS:
        files.extend(folder.glob(pattern))
    return sorted(set(files))

def load_image_rgb(path: Path) -> Image.Image:
    return Image.open(path).convert("RGB")

def show_input_images(reference_paths, observed_paths, max_reference=3, max_observed=3):
    items = []
    for p in reference_paths[:max_reference]:
        items.append((f"Reference: {p.name}", load_image_rgb(p)))
    for p in observed_paths[:max_observed]:
        items.append((f"Observed: {p.name}", load_image_rgb(p)))

    if not items:
        return

    fig, axes = plt.subplots(1, len(items), figsize=(4 * len(items), 4))
    if len(items) == 1:
        axes = [axes]

    for ax, (title, image) in zip(axes, items):
        ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

reference_paths = list_images(REFERENCE_DIR)
observed_paths = list_images(OBSERVED_DIR)

print("Reference images:")
for p in reference_paths:
    print(" -", p.name)

print("\nObserved images:")
for p in observed_paths:
    print(" -", p.name)

if not reference_paths:
    print("\nWARNING: data/reference_normal/ に正常画像を入れてください")
if not observed_paths:
    print("\nWARNING: data/observed/ に検査画像を入れてください")

show_input_images(reference_paths, observed_paths)

## 4. Hugging Faceトークン入力

DINOv3の一部モデルは gated repo です。Hugging Face上のトークン名、たとえば `hugging_token` は管理用の名前なので、コードには入れません。入力するのはトークン値だけです。

入力したトークンはNotebookファイルには保存せず、このカーネル内の `HF_TOKEN` として使います。

In [ ]:
from getpass import getpass
from huggingface_hub import whoami

def ensure_hf_token():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

    if token:
        whoami(token=token)
        os.environ["HF_TOKEN"] = token
        print("Hugging Face token is available.")
        return token

    token = getpass("Hugging Face token（入力は表示されません）: ").strip()
    if not token:
        raise ValueError("Hugging Face token が入力されていません")

    whoami(token=token)
    os.environ["HF_TOKEN"] = token
    print("Hugging Face token is available.")
    return token

HF_TOKEN = ensure_hf_token()

## 5. DINOv3設定

本線はDINOv3です。DINOv2は最後の付録でのみ使います。

`ANOMALY_SCORE_THRESHOLD` は論文寄りのraw anomaly scoreに対するしきい値です。`VISUAL_THRESHOLD` ではありません。`USE_ROI` は机上の領域だけを見たい場合に使います。初期値では画像端とキーボード下部を抑えます。

In [ ]:
MODEL_NAME = "facebook/dinov3-vits16-pretrain-lvd1689m"

# ViT-S/16なら 448 / 16 = 28 なので、28x28の異常マップになります。
IMAGE_SIZE = 448
REFERENCE_CHUNK_SIZE = 16384

# raw anomaly score = (1 - cosine_similarity) / 2 に対するしきい値。
# 0.18では候補0件になりやすい場合があるため、初期値は少し低めにしています。
ANOMALY_SCORE_THRESHOLD = 0.10

# 細かいノイズ除去と候補表示。
MIN_COMPONENT_AREA = 120
MAX_COMPONENTS = 10
TOP_SCORE_POINTS = 10
TOP_POINT_MIN_DISTANCE = 24

# 机の中央付近だけを見るROI。キーボードや画像端の反応を抑えるための初期値です。
USE_ROI = True
ROI_X_MIN = 0.04
ROI_X_MAX = 0.96
ROI_Y_MIN = 0.04
ROI_Y_MAX = 0.72

print("MODEL_NAME:", MODEL_NAME)
print("IMAGE_SIZE:", IMAGE_SIZE)
print("ANOMALY_SCORE_THRESHOLD:", ANOMALY_SCORE_THRESHOLD)
print("MIN_COMPONENT_AREA:", MIN_COMPONENT_AREA)
print("USE_ROI:", USE_ROI)

## 6. DINOv3モデル読み込み

Hugging Faceの `HF_TOKEN` を使ってDINOv3を読み込みます。ここで403が出る場合は、トークンは有効でも、そのアカウントがDINOv3モデルの許可リストに入っていません。

In [ ]:
from transformers import AutoModel

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if not token:
    raise ValueError("HF_TOKEN がありません。前のHugging Faceトークン入力セルを実行してください。")

try:
    model = AutoModel.from_pretrained(MODEL_NAME, token=token)
except OSError:
    print("DINOv3モデルを読み込めませんでした。")
    print("確認してください:")
    print("1. 入力したトークン値が正しいか")
    print("2. https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m でアクセス許可が通っているか")
    print("3. このNotebookを動かしているカーネルにトークンを入力したか")
    raise

ACTIVE_MODEL_NAME = MODEL_NAME
model = model.to(DEVICE)
model.eval()

patch_size = model.config.patch_size
num_register_tokens = getattr(model.config, "num_register_tokens", 0)
hidden_size = model.config.hidden_size

print("ACTIVE_MODEL_NAME:", ACTIVE_MODEL_NAME)
print("patch_size:", patch_size)
print("num_register_tokens:", num_register_tokens)
print("hidden_size:", hidden_size)

## 7. 画像前処理

ImageNet系の平均・標準偏差で正規化します。

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])

def image_to_tensor(image: Image.Image) -> torch.Tensor:
    return transform(image).unsqueeze(0).to(DEVICE)

## 8. パッチ特徴抽出

ViTの出力から `CLS token + register tokens` を除き、patch tokenだけを使います。

In [ ]:
@torch.inference_mode()
def extract_patch_features(image_path: Path):
    image = load_image_rgb(image_path)
    image_resized = image.resize((IMAGE_SIZE, IMAGE_SIZE))
    pixel_values = image_to_tensor(image)

    outputs = model(pixel_values=pixel_values)
    last_hidden = outputs.last_hidden_state

    _, _, img_h, img_w = pixel_values.shape
    grid_h = img_h // patch_size
    grid_w = img_w // patch_size
    num_patches = grid_h * grid_w

    special_tokens = 1 + num_register_tokens
    patch_tokens = last_hidden[:, special_tokens:, :]

    assert patch_tokens.shape[1] == num_patches, (
        f"patch数が合いません: got={patch_tokens.shape[1]}, expected={num_patches}"
    )

    patch_tokens = patch_tokens.squeeze(0)
    patch_tokens = F.normalize(patch_tokens, p=2, dim=-1)

    return {
        "features": patch_tokens,
        "grid_h": grid_h,
        "grid_w": grid_w,
        "image": image_resized,
        "path": image_path,
    }

## 9. 正常特徴バンク作成

論文のsingle-reference設定に合わせるなら、`reference_normal/` にはまず1枚だけ入れてください。ここで、どのモデルで作った特徴バンクかも記録して、DINOv2付録との混線を防ぎます。

In [ ]:
reference_paths = list_images(REFERENCE_DIR)
if not reference_paths:
    raise ValueError("data/reference_normal/ に正常画像を入れてください")

reference_feature_list = []
for path in tqdm(reference_paths, desc="Extracting reference features"):
    item = extract_patch_features(path)
    reference_feature_list.append(item["features"])

reference_bank = torch.cat(reference_feature_list, dim=0).contiguous()
reference_bank_model_name = ACTIVE_MODEL_NAME
reference_bank_image_size = IMAGE_SIZE
reference_bank_hidden_size = hidden_size
reference_bank_patch_size = patch_size

print("num reference images:", len(reference_paths))
print("reference_bank shape:", tuple(reference_bank.shape))
print("reference_bank_model_name:", reference_bank_model_name)

In [ ]:
def save_reference_bank(reference_bank: torch.Tensor):
    safe_model = ACTIVE_MODEL_NAME.replace("/", "__")
    path = CACHE_DIR / f"reference_bank_{safe_model}_{IMAGE_SIZE}.pt"
    metadata_path = CACHE_DIR / f"reference_bank_{safe_model}_{IMAGE_SIZE}.json"

    torch.save(reference_bank.detach().cpu(), path)
    metadata = {
        "model_name": ACTIVE_MODEL_NAME,
        "image_size": IMAGE_SIZE,
        "patch_size": patch_size,
        "hidden_size": hidden_size,
        "reference_images": [p.name for p in reference_paths],
    }
    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    return path, metadata_path

bank_path, metadata_path = save_reference_bank(reference_bank)
print("saved:", bank_path.relative_to(PROJECT_ROOT))
print("saved:", metadata_path.relative_to(PROJECT_ROOT))

## 10. 異常マップ計算

論文寄りに、raw anomaly scoreは `((1 - cosine_similarity) / 2)` として `[0, 1]` に写像します。

`anomaly_raw` は定量値とmask用、`anomaly_visual` はheatmapを見やすくするための表示用です。

In [ ]:
def validate_reference_bank(reference_bank: torch.Tensor):
    if reference_bank_model_name != ACTIVE_MODEL_NAME:
        raise ValueError(
            f"reference_bank は {reference_bank_model_name} で作られていますが、"
            f"現在のmodelは {ACTIVE_MODEL_NAME} です。特徴空間が混ざるため再作成してください。"
        )
    if reference_bank_image_size != IMAGE_SIZE:
        raise ValueError(
            f"reference_bank は IMAGE_SIZE={reference_bank_image_size} で作られていますが、"
            f"現在は IMAGE_SIZE={IMAGE_SIZE} です。"
        )
    if reference_bank.shape[1] != hidden_size:
        raise ValueError(
            f"reference_bank hidden size={reference_bank.shape[1]} と現在のmodel hidden size={hidden_size} が一致しません。"
        )

def robust_normalize(x: torch.Tensor, low_q=0.02, high_q=0.98):
    x_flat = x.flatten()
    lo = torch.quantile(x_flat, low_q)
    hi = torch.quantile(x_flat, high_q)

    if float(hi - lo) < 1e-6:
        return torch.zeros_like(x)

    x = (x - lo) / (hi - lo)
    return x.clamp(0, 1)

def max_cosine_similarity_chunked(query_features: torch.Tensor, bank_features: torch.Tensor, chunk_size=16384):
    best = None
    for start in range(0, bank_features.shape[0], chunk_size):
        bank_chunk = bank_features[start:start + chunk_size]
        sim = query_features @ bank_chunk.T
        chunk_best = sim.max(dim=1).values
        best = chunk_best if best is None else torch.maximum(best, chunk_best)
    return best

@torch.inference_mode()
def compute_anomaly_map(observed_path: Path, reference_bank: torch.Tensor):
    validate_reference_bank(reference_bank)

    obs = extract_patch_features(observed_path)
    observed_features = obs["features"]

    best_sim = max_cosine_similarity_chunked(
        observed_features,
        reference_bank,
        chunk_size=REFERENCE_CHUNK_SIZE,
    )

    anomaly_score = ((1.0 - best_sim) / 2.0).clamp(0, 1)
    anomaly_grid = anomaly_score.reshape(obs["grid_h"], obs["grid_w"])
    anomaly_visual = robust_normalize(anomaly_grid)

    return {
        "observed": obs,
        "anomaly_raw": anomaly_grid.detach().cpu(),
        "anomaly_visual": anomaly_visual.detach().cpu(),
        "best_sim": best_sim.detach().cpu(),
    }

## 11. ROI・候補分割・可視化

「どれのどこが異常か」を分かるように、maskをconnected componentsで分割し、各候補のbbox・中心・面積・スコアを出します。

In [ ]:
def tensor_grid_to_numpy_map(grid: torch.Tensor, size=(IMAGE_SIZE, IMAGE_SIZE)):
    x = grid.unsqueeze(0).unsqueeze(0).float()
    x_up = F.interpolate(x, size=size, mode="bicubic", align_corners=False)
    return x_up.squeeze().numpy().clip(0, 1)

def build_roi_mask(shape):
    h, w = shape
    roi = np.zeros((h, w), dtype=bool)
    x0 = int(w * ROI_X_MIN)
    x1 = int(w * ROI_X_MAX)
    y0 = int(h * ROI_Y_MIN)
    y1 = int(h * ROI_Y_MAX)
    roi[y0:y1, x0:x1] = True
    return roi

def apply_roi(score_np):
    if not USE_ROI:
        return score_np.copy()
    roi = build_roi_mask(score_np.shape)
    masked = score_np.copy()
    masked[~roi] = 0
    return masked

def make_heatmap(anomaly_map_np, cmap_name="magma"):
    cmap = colormaps[cmap_name]
    heatmap = cmap(anomaly_map_np)[:, :, :3]
    heatmap = (heatmap * 255).astype(np.uint8)
    return Image.fromarray(heatmap)

def make_overlay(image: Image.Image, heatmap: Image.Image, alpha=0.45):
    return Image.blend(image.convert("RGB"), heatmap.convert("RGB"), alpha=alpha)

def make_mask_image(mask_bool):
    return Image.fromarray(mask_bool.astype(np.uint8) * 255, mode="L")

def clean_binary_mask(mask_bool, min_area=MIN_COMPONENT_AREA):
    labeled, num = ndimage.label(mask_bool)
    cleaned = np.zeros_like(mask_bool, dtype=bool)

    for label_id in range(1, num + 1):
        component = labeled == label_id
        if int(component.sum()) >= min_area:
            cleaned[component] = True

    return cleaned

def get_components(mask_bool, score_np=None, max_components=MAX_COMPONENTS):
    labeled, num = ndimage.label(mask_bool)
    components = []

    for label_id in range(1, num + 1):
        component = labeled == label_id
        area = int(component.sum())
        if area == 0:
            continue

        ys, xs = np.where(component)
        row = {
            "rank": 0,
            "label_id": int(label_id),
            "center_x": float(xs.mean()),
            "center_y": float(ys.mean()),
            "area": area,
            "area_ratio": float(area / mask_bool.size),
            "x_min": int(xs.min()),
            "x_max": int(xs.max()),
            "y_min": int(ys.min()),
            "y_max": int(ys.max()),
            "decision": "accepted_component",
        }

        if score_np is not None:
            scores = score_np[component]
            row["mean_score"] = float(scores.mean())
            row["max_score"] = float(scores.max())

        components.append(row)

    components.sort(key=lambda x: (x.get("max_score", 0), x["area"]), reverse=True)
    components = components[:max_components]
    for i, row in enumerate(components, start=1):
        row["rank"] = i

    return components

def get_top_score_points(score_np, k=TOP_SCORE_POINTS, min_distance=TOP_POINT_MIN_DISTANCE):
    h, w = score_np.shape
    flat_order = np.argsort(score_np.reshape(-1))[::-1]
    points = []

    for flat_idx in flat_order:
        y, x = divmod(int(flat_idx), w)
        score = float(score_np[y, x])
        if score <= 0:
            break

        too_close = False
        for p in points:
            if ((p["center_x"] - x) ** 2 + (p["center_y"] - y) ** 2) ** 0.5 < min_distance:
                too_close = True
                break
        if too_close:
            continue

        r = 10
        points.append({
            "rank": len(points) + 1,
            "center_x": float(x),
            "center_y": float(y),
            "area": 0,
            "area_ratio": 0.0,
            "x_min": max(0, int(x - r)),
            "y_min": max(0, int(y - r)),
            "x_max": min(w - 1, int(x + r)),
            "y_max": min(h - 1, int(y + r)),
            "mean_score": score,
            "max_score": score,
            "decision": "top_score_point_not_component",
        })
        if len(points) >= k:
            break

    return points

def make_candidate_dataframe(components, top_points):
    columns = [
        "decision", "rank", "center_x", "center_y", "area", "area_ratio",
        "x_min", "y_min", "x_max", "y_max", "mean_score", "max_score",
    ]
    rows = []

    for row in components:
        rows.append({key: row.get(key, np.nan) for key in columns})

    if not rows:
        for row in top_points:
            rows.append({key: row.get(key, np.nan) for key in columns})

    return pd.DataFrame(rows, columns=columns)

def print_anomaly_decision(observed_name, components, top_points, threshold=ANOMALY_SCORE_THRESHOLD):
    max_point = top_points[0] if top_points else None
    print(f"observed: {observed_name}")
    print(f"raw threshold: {threshold}")
    print(f"min component area: {MIN_COMPONENT_AREA}")
    print(f"accepted anomaly components: {len(components)}")

    if components:
        for comp in components:
            print(
                f"異常候補 #{comp['rank']}: "
                f"center=({comp['center_x']:.1f}, {comp['center_y']:.1f}), "
                f"bbox=({comp['x_min']},{comp['y_min']})-({comp['x_max']},{comp['y_max']}), "
                f"area={comp['area']}, max_score={comp.get('max_score', float('nan')):.4f}"
            )
    elif max_point:
        print("採用された異常候補は0件です。")
        print(
            "ただし最大raw score地点は "
            f"({max_point['center_x']:.1f}, {max_point['center_y']:.1f}) "
            f"score={max_point['max_score']:.4f} です。"
        )
        print("この地点を異常として採用したい場合は、ANOMALY_SCORE_THRESHOLDを下げるかMIN_COMPONENT_AREAを下げてください。")

def annotate_anomalies(image: Image.Image, components, top_points=None, color=(255, 40, 40)):
    annotated = image.convert("RGB").copy()
    draw = ImageDraw.Draw(annotated)

    for comp in components:
        box = [comp["x_min"], comp["y_min"], comp["x_max"], comp["y_max"]]
        label = f"#{comp['rank']} score={comp.get('max_score', 0):.3f} area={comp['area']}"
        draw.rectangle(box, outline=color, width=4)
        text_xy = (comp["x_min"] + 4, max(0, comp["y_min"] - 24))
        draw.rectangle([text_xy[0] - 2, text_xy[1] - 2, text_xy[0] + 260, text_xy[1] + 22], fill=(255, 255, 255))
        draw.text(text_xy, label, fill=color)

    if not components and top_points:
        point_color = (40, 120, 255)
        for p in top_points[:5]:
            x = int(p["center_x"])
            y = int(p["center_y"])
            r = 12
            draw.ellipse([x - r, y - r, x + r, y + r], outline=point_color, width=4)
            draw.line([x - r, y, x + r, y], fill=point_color, width=3)
            draw.line([x, y - r, x, y + r], fill=point_color, width=3)
            label = f"max#{p['rank']} {p['max_score']:.3f}"
            draw.rectangle([x + 6, max(0, y - 24), x + 150, max(0, y - 2)], fill=(255, 255, 255))
            draw.text((x + 8, max(0, y - 24)), label, fill=point_color)

    return annotated

def show_result(reference_image, observed_image, heatmap, overlay, annotated, mask):
    fig, axes = plt.subplots(1, 6, figsize=(26, 5))
    titles = ["Reference normal", "Observed", "Anomaly heatmap", "Overlay", "Annotated", "Mask"]
    images = [reference_image, observed_image, heatmap, overlay, annotated, mask]
    cmaps = [None, None, None, None, None, "gray"]

    for ax, title, img, cmap in zip(axes, titles, images, cmaps):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

def show_result_separately(reference_image, observed_image, heatmap, overlay, annotated, mask, title_prefix=""):
    items = [
        ("Reference normal", reference_image, None),
        ("Observed", observed_image, None),
        ("Anomaly heatmap", heatmap, None),
        ("Overlay", overlay, None),
        ("Annotated", annotated, None),
        ("Mask", mask, "gray"),
    ]

    for title, image, cmap in items:
        fig, ax = plt.subplots(1, 1, figsize=(8, 8))
        ax.imshow(image, cmap=cmap)
        ax.set_title(f"{title_prefix}{title}")
        ax.axis("off")
        plt.tight_layout()
        plt.show()

def analyze_observed_image(path: Path, reference_bank: torch.Tensor):
    result = compute_anomaly_map(path, reference_bank)
    anomaly_raw_np = tensor_grid_to_numpy_map(result["anomaly_raw"], size=(IMAGE_SIZE, IMAGE_SIZE))
    anomaly_visual_np = tensor_grid_to_numpy_map(result["anomaly_visual"], size=(IMAGE_SIZE, IMAGE_SIZE))

    anomaly_raw_roi_np = apply_roi(anomaly_raw_np)
    anomaly_visual_roi_np = apply_roi(anomaly_visual_np)

    pre_clean_mask_bool = anomaly_raw_roi_np >= ANOMALY_SCORE_THRESHOLD
    mask_bool = clean_binary_mask(pre_clean_mask_bool, min_area=MIN_COMPONENT_AREA)
    components = get_components(mask_bool, score_np=anomaly_raw_roi_np)
    top_points = get_top_score_points(anomaly_raw_roi_np)
    candidates_df = make_candidate_dataframe(components, top_points)

    observed_image = result["observed"]["image"]
    reference_image = load_image_rgb(reference_paths[0]).resize((IMAGE_SIZE, IMAGE_SIZE))
    heatmap = make_heatmap(anomaly_visual_roi_np)
    overlay = make_overlay(observed_image, heatmap, alpha=0.45)
    mask = make_mask_image(mask_bool)
    annotated = annotate_anomalies(observed_image, components, top_points=top_points)

    return {
        "result": result,
        "anomaly_raw_np": anomaly_raw_np,
        "anomaly_visual_np": anomaly_visual_np,
        "anomaly_raw_roi_np": anomaly_raw_roi_np,
        "anomaly_visual_roi_np": anomaly_visual_roi_np,
        "pre_clean_mask_bool": pre_clean_mask_bool,
        "mask_bool": mask_bool,
        "components": components,
        "top_points": top_points,
        "candidates_df": candidates_df,
        "reference_image": reference_image,
        "observed_image": observed_image,
        "heatmap": heatmap,
        "overlay": overlay,
        "mask": mask,
        "annotated": annotated,
    }

## 論文そのものではない追加処理

このNotebookの中心部分は、論文のコア手法に寄せています。

```text
DINOv3特徴
single reference
patch token
L2正規化
最近傍cos類似度
anomaly score
upsampling
threshold mask
```

ただし、Notebookには机上実験で見やすく・使いやすくするための追加処理があります。以下は論文そのままの最小手法ではなく、この実験用の後処理です。

### 1. ROI処理

このNotebookでは、初期状態で次のROIを使います。

```python
USE_ROI = True
ROI_X_MIN = 0.04
ROI_X_MAX = 0.96
ROI_Y_MIN = 0.04
ROI_Y_MAX = 0.72
```

これは、机上の中央付近だけを見るための設定です。キーボード、画像端、背景境界の誤反応を抑えられます。

論文のコアは次です。

```text
DINOv3特徴 + single reference + 最近傍cos類似度 + anomaly score
```

このNotebookの追加は次です。

```text
ROIで見たい範囲だけに制限
```

これは論文そのままではありませんが、今回の机上実験では妥当な実用上の工夫です。

### 2. connected components

このNotebookでは、maskを作ったあとに白領域を候補ごとに分けます。

```text
bbox
中心座標
面積
mean_score
max_score
```

を出すためです。

論文はdense anomaly maskを出します。一方、このNotebookの用途では、ロボットやカメラ制御で「どこへ向かえばよいか」「どの物体を見るべきか」を知りたいので、connected componentsで候補化しています。

これも論文の中心処理そのものではなく、利用しやすくするための追加後処理です。

### 3. thresholdは実験値

現在の初期値は次です。

```python
ANOMALY_SCORE_THRESHOLD = 0.10
```

この値は、今回の机画像セットではうまく動いた値です。ただし普遍的な値ではありません。照明、カメラ位置、机の模様、画像サイズ、reference画像が変わると、適切なしきい値も変わります。

そのため、このNotebookでは「raw scoreしきい値確認」セルを入れています。結果を見ながら `ANOMALY_SCORE_THRESHOLD` を調整してください。

### 4. 異常スコアは確率ではない

たとえば `max_score = 0.2139` は、21.39%の確率で異常、という意味ではありません。

意味は次です。

```text
DINOv3特徴空間で、正常referenceのどのpatchにもあまり似ていない度合い
```

流れとしてはこうです。

```text
cos類似度が低い
↓
reference_normalには似た見た目・意味のpatchが少ない
↓
異常スコアが高い
```

同じカメラ位置、同じ机、近い照明条件で比較するほど、このスコアは扱いやすくなります。

## 12. 1枚だけ試す

まず `observed/` の先頭画像で、異常候補のbbox・中心座標・スコアを確認します。

In [ ]:
observed_paths = list_images(OBSERVED_DIR)
if not observed_paths:
    raise ValueError("data/observed/ に検査画像を入れてください")

target_path = observed_paths[0]
print("target:", target_path.name)

analysis = analyze_observed_image(target_path, reference_bank)

reference_image = analysis["reference_image"]
observed_image = analysis["observed_image"]
heatmap = analysis["heatmap"]
overlay = analysis["overlay"]
mask = analysis["mask"]
annotated = analysis["annotated"]
components = analysis["components"]
top_points = analysis["top_points"]
candidates_df = analysis["candidates_df"]
anomaly_raw_roi_np = analysis["anomaly_raw_roi_np"]
anomaly_visual_roi_np = analysis["anomaly_visual_roi_np"]
mask_bool = analysis["mask_bool"]

show_result(reference_image, observed_image, heatmap, overlay, annotated, mask)
show_result_separately(reference_image, observed_image, heatmap, overlay, annotated, mask, title_prefix="DINOv3 - ")
print_anomaly_decision(target_path.name, components, top_points)
display(candidates_df)

## 13. raw scoreしきい値確認

maskは `anomaly_raw` から作っています。しきい値を変えると候補がどう変わるか確認します。

In [ ]:
thresholds = [0.04, 0.06, 0.08, 0.10, 0.14, 0.18]
fig, axes = plt.subplots(1, len(thresholds), figsize=(4 * len(thresholds), 4))

for ax, th in zip(axes, thresholds):
    m = clean_binary_mask(anomaly_raw_roi_np >= th, min_area=MIN_COMPONENT_AREA)
    comps = get_components(m, score_np=anomaly_raw_roi_np)
    ax.imshow(make_mask_image(m), cmap="gray")
    ax.set_title(f"th={th}\ncomponents={len(comps)}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 14. 結果保存

`heatmap`, `overlay`, `mask`, `annotated`, `comparison`, `components.csv` を保存します。

In [ ]:
def save_comparison_image(comparison_path: Path, reference_image, observed_image, heatmap, overlay, annotated, mask):
    fig, axes = plt.subplots(1, 6, figsize=(26, 5))
    titles = ["Reference normal", "Observed", "Anomaly heatmap", "Overlay", "Annotated", "Mask"]
    images = [reference_image, observed_image, heatmap, overlay, annotated, mask]
    cmaps = [None, None, None, None, None, "gray"]

    for ax, title, img, cmap in zip(axes, titles, images, cmaps):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    fig.savefig(comparison_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

def save_outputs(observed_path: Path, reference_image, observed_image, heatmap, overlay, annotated, mask, candidates_df):
    stem = observed_path.stem
    heatmap_path = HEATMAP_DIR / f"{stem}_heatmap.png"
    overlay_path = OVERLAY_DIR / f"{stem}_overlay.png"
    mask_path = MASK_DIR / f"{stem}_mask.png"
    annotated_path = ANNOTATED_DIR / f"{stem}_annotated.png"
    comparison_path = COMPARISON_DIR / f"{stem}_comparison.png"
    components_path = COMPONENTS_DIR / f"{stem}_candidates.csv"

    heatmap.save(heatmap_path)
    overlay.save(overlay_path)
    mask.save(mask_path)
    annotated.save(annotated_path)
    candidates_df.to_csv(components_path, index=False, encoding="utf-8-sig")
    save_comparison_image(comparison_path, reference_image, observed_image, heatmap, overlay, annotated, mask)

    return {
        "heatmap": str(heatmap_path.relative_to(PROJECT_ROOT)),
        "overlay": str(overlay_path.relative_to(PROJECT_ROOT)),
        "mask": str(mask_path.relative_to(PROJECT_ROOT)),
        "annotated": str(annotated_path.relative_to(PROJECT_ROOT)),
        "comparison": str(comparison_path.relative_to(PROJECT_ROOT)),
        "candidates": str(components_path.relative_to(PROJECT_ROOT)),
    }

save_outputs(target_path, reference_image, observed_image, heatmap, overlay, annotated, mask, candidates_df)

## 15. observed内を一括処理

すべての検査画像を処理します。`summary.csv` には最大候補の中心座標とbboxを入れ、各画像の全候補は `components/` に個別CSVとして保存します。

In [ ]:
rows = []

for path in tqdm(observed_paths, desc="Processing observed images"):
    analysis = analyze_observed_image(path, reference_bank)

    paths = save_outputs(
        path,
        analysis["reference_image"],
        analysis["observed_image"],
        analysis["heatmap"],
        analysis["overlay"],
        analysis["annotated"],
        analysis["mask"],
        analysis["candidates_df"],
    )

    components = analysis["components"]
    top_points = analysis["top_points"]
    top = components[0] if components else (top_points[0] if top_points else {})

    rows.append({
        "observed": path.name,
        "model_name": ACTIVE_MODEL_NAME,
        "reference_bank_model_name": reference_bank_model_name,
        "anomaly_raw_mean": float(analysis["anomaly_raw_roi_np"].mean()),
        "anomaly_raw_max": float(analysis["anomaly_raw_roi_np"].max()),
        "mask_area_ratio": float(analysis["mask_bool"].mean()),
        "component_count": len(components),
        "top_decision": top.get("decision", "none"),
        "top_center_x": top.get("center_x", np.nan),
        "top_center_y": top.get("center_y", np.nan),
        "top_area": top.get("area", 0),
        "top_area_ratio": top.get("area_ratio", 0.0),
        "top_x_min": top.get("x_min", np.nan),
        "top_y_min": top.get("y_min", np.nan),
        "top_x_max": top.get("x_max", np.nan),
        "top_y_max": top.get("y_max", np.nan),
        "top_max_score": top.get("max_score", np.nan),
        **paths,
    })

summary_df = pd.DataFrame(rows)
summary_df

In [ ]:
summary_path = OUTPUT_DIR / "summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
print("saved:", summary_path.relative_to(PROJECT_ROOT))

## 付録: DINOv2でも同じ処理を試す

本線はDINOv3です。この付録は比較用です。必要な場合だけ次のセルで `RUN_DINOV2_APPENDIX = True` にして実行してください。

出力先は `outputs/desk_anomaly_dino_appendix_dinov2/` です。DINOv3本線の `outputs/desk_anomaly_dino/` は上書きしません。付録実行後は、DINOv3の `model` と `reference_bank` を必ず復元します。

In [ ]:
RUN_DINOV2_APPENDIX = True

if not RUN_DINOV2_APPENDIX:
    print("DINOv2付録は未実行です。実行する場合は RUN_DINOV2_APPENDIX = True にしてください。")
else:
    saved = {
        "model": model,
        "active_model_name": ACTIVE_MODEL_NAME,
        "patch_size": patch_size,
        "num_register_tokens": num_register_tokens,
        "hidden_size": hidden_size,
        "output_dir": OUTPUT_DIR,
        "heatmap_dir": HEATMAP_DIR,
        "overlay_dir": OVERLAY_DIR,
        "mask_dir": MASK_DIR,
        "annotated_dir": ANNOTATED_DIR,
        "comparison_dir": COMPARISON_DIR,
        "components_dir": COMPONENTS_DIR,
        "reference_bank": reference_bank,
        "reference_bank_model_name": reference_bank_model_name,
        "reference_bank_image_size": reference_bank_image_size,
        "reference_bank_hidden_size": reference_bank_hidden_size,
        "reference_bank_patch_size": reference_bank_patch_size,
    }

    DINO2_MODEL_NAME = "facebook/dinov2-small"
    OUTPUT_DIR = PROJECT_ROOT / "outputs" / "desk_anomaly_dino_appendix_dinov2"
    HEATMAP_DIR = OUTPUT_DIR / "heatmap"
    OVERLAY_DIR = OUTPUT_DIR / "overlay"
    MASK_DIR = OUTPUT_DIR / "mask"
    ANNOTATED_DIR = OUTPUT_DIR / "annotated"
    COMPARISON_DIR = OUTPUT_DIR / "comparison"
    COMPONENTS_DIR = OUTPUT_DIR / "components"
    for d in [HEATMAP_DIR, OVERLAY_DIR, MASK_DIR, ANNOTATED_DIR, COMPARISON_DIR, COMPONENTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    model = AutoModel.from_pretrained(DINO2_MODEL_NAME).to(DEVICE)
    model.eval()
    ACTIVE_MODEL_NAME = DINO2_MODEL_NAME
    patch_size = model.config.patch_size
    num_register_tokens = getattr(model.config, "num_register_tokens", 0)
    hidden_size = model.config.hidden_size

    reference_feature_list = []
    for path in tqdm(reference_paths, desc="Extracting DINOv2 reference features"):
        item = extract_patch_features(path)
        reference_feature_list.append(item["features"])
    reference_bank = torch.cat(reference_feature_list, dim=0).contiguous()
    reference_bank_model_name = ACTIVE_MODEL_NAME
    reference_bank_image_size = IMAGE_SIZE
    reference_bank_hidden_size = hidden_size
    reference_bank_patch_size = patch_size

    rows = []
    first_analysis = None
    first_path = None

    for path in tqdm(observed_paths, desc="Processing observed images with DINOv2"):
        analysis = analyze_observed_image(path, reference_bank)
        if first_analysis is None:
            first_analysis = analysis
            first_path = path

        paths = save_outputs(
            path,
            analysis["reference_image"],
            analysis["observed_image"],
            analysis["heatmap"],
            analysis["overlay"],
            analysis["annotated"],
            analysis["mask"],
            analysis["candidates_df"],
        )

        components = analysis["components"]
        top_points = analysis["top_points"]
        top = components[0] if components else (top_points[0] if top_points else {})

        rows.append({
            "observed": path.name,
            "model_name": ACTIVE_MODEL_NAME,
            "reference_bank_model_name": reference_bank_model_name,
            "anomaly_raw_mean": float(analysis["anomaly_raw_roi_np"].mean()),
            "anomaly_raw_max": float(analysis["anomaly_raw_roi_np"].max()),
            "mask_area_ratio": float(analysis["mask_bool"].mean()),
            "component_count": len(components),
            "top_decision": top.get("decision", "none"),
            "top_center_x": top.get("center_x", np.nan),
            "top_center_y": top.get("center_y", np.nan),
            "top_area": top.get("area", 0),
            "top_area_ratio": top.get("area_ratio", 0.0),
            "top_max_score": top.get("max_score", np.nan),
            **paths,
        })

    summary_df_dinov2 = pd.DataFrame(rows)
    summary_path = OUTPUT_DIR / "summary.csv"
    summary_df_dinov2.to_csv(summary_path, index=False, encoding="utf-8-sig")
    print("saved:", summary_path.relative_to(PROJECT_ROOT))
    display(summary_df_dinov2)

    if first_analysis is not None:
        print("DINOv2 first observed visualization:", first_path.name)
        show_result(
            first_analysis["reference_image"],
            first_analysis["observed_image"],
            first_analysis["heatmap"],
            first_analysis["overlay"],
            first_analysis["annotated"],
            first_analysis["mask"],
        )
        show_result_separately(
            first_analysis["reference_image"],
            first_analysis["observed_image"],
            first_analysis["heatmap"],
            first_analysis["overlay"],
            first_analysis["annotated"],
            first_analysis["mask"],
            title_prefix="DINOv2 - ",
        )
        print_anomaly_decision(first_path.name, first_analysis["components"], first_analysis["top_points"])
        display(first_analysis["candidates_df"])

    model = saved["model"]
    ACTIVE_MODEL_NAME = saved["active_model_name"]
    patch_size = saved["patch_size"]
    num_register_tokens = saved["num_register_tokens"]
    hidden_size = saved["hidden_size"]
    OUTPUT_DIR = saved["output_dir"]
    HEATMAP_DIR = saved["heatmap_dir"]
    OVERLAY_DIR = saved["overlay_dir"]
    MASK_DIR = saved["mask_dir"]
    ANNOTATED_DIR = saved["annotated_dir"]
    COMPARISON_DIR = saved["comparison_dir"]
    COMPONENTS_DIR = saved["components_dir"]
    reference_bank = saved["reference_bank"]
    reference_bank_model_name = saved["reference_bank_model_name"]
    reference_bank_image_size = saved["reference_bank_image_size"]
    reference_bank_hidden_size = saved["reference_bank_hidden_size"]
    reference_bank_patch_size = saved["reference_bank_patch_size"]